# SDXL × DrawBench — Standard Sampling (Colab)

Generates standard-sampling images for SDXL on all 200 DrawBench prompts.
Requires a Colab

In [1]:
!pip install -q diffusers transformers accelerate safetensors

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import gc
import json
import shutil
import time
from pathlib import Path

import torch
from diffusers import DDIMScheduler, StableDiffusionXLPipeline
from IPython.display import display
from PIL import Image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"torch: {torch.__version__}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Device: cuda
torch: 2.11.0+cu128
GPU: NVIDIA A100-SXM4-40GB


## Config

In [4]:
MODEL_ID = "stabilityai/stable-diffusion-xl-base-1.0"
NUM_INFERENCE_STEPS = 50
GUIDANCE_SCALE = 5.5
HEIGHT = 1024
WIDTH = 1024
SEEDS = [2026]

OUTPUT_DIR = Path("/content/outputs/daniel_sdxl_drawbench")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DRIVE_DIR = Path("/content/drive/MyDrive/generative-modelling/sdxl_drawbench")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_EVERY = 25

# How many prompts to run. Set to None for all 200.
NUM_PROMPTS = None
PROMPT_SEED = 42

## DrawBench Prompts (200)

In [5]:
import random
import pandas as pd

CSV_URL = "https://huggingface.co/datasets/sayakpaul/drawbench/raw/main/DrawBench%20Prompts%20-%20Sheet1.csv"
df = pd.read_csv(CSV_URL)
DRAWBENCH_PROMPTS = list(enumerate(df["Prompts"].tolist()))

if NUM_PROMPTS is not None and NUM_PROMPTS < len(DRAWBENCH_PROMPTS):
    rng = random.Random(PROMPT_SEED)
    selected_prompts = sorted(rng.sample(DRAWBENCH_PROMPTS, NUM_PROMPTS))
    print(f"Randomly selected {NUM_PROMPTS}/{len(DRAWBENCH_PROMPTS)} prompts (seed={PROMPT_SEED})")
else:
    selected_prompts = DRAWBENCH_PROMPTS
    print(f"Running all {len(selected_prompts)} prompts")

print(f"First 5: {[f'{i}: {p[:40]}...' for i, p in selected_prompts[:5]]}")
df.head()

Running all 200 prompts
First 5: ['0: A red colored car....', '1: A black colored car....', '2: A pink colored car....', '3: A black colored dog....', '4: A red colored dog....']


,Prompts,Category
0,A red colored car.,Colors
1,A black colored car.,Colors
2,A pink colored car.,Colors
3,A black colored dog.,Colors
4,A red colored dog.,Colors


In [12]:
selected_prompts = selected_prompts[150:]

## Load Pipeline

In [9]:
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID, torch_dtype=DTYPE, use_safetensors=True
)
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to(DEVICE)
if DEVICE == "cuda":
    pipe.enable_attention_slicing()
    pipe.enable_vae_tiling()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


model_index.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/diffusers/pipelines/pipeline_utils.py:2290: FutureWarning: `enable_vae_tiling` is deprecated and will be removed in version 0.40.0. Calling `enable_vae_tiling()` on a `StableDiffusionXLPipeline` is deprecated and this method will be removed in a future version. Please use `pipe.vae.enable_tiling()`.
  deprecate(


## Generate

Code duplicated from /src

In [14]:
import os
import tempfile

def checkpoint_to_drive(files, tag):
    """Zip only the given files and copy to Drive with flush."""
    with tempfile.TemporaryDirectory() as tmp:
        for f in files:
            shutil.copy2(f, Path(tmp) / f.name)
        zp = shutil.make_archive(f"/content/{tag}", "zip", tmp)
    dest = DRIVE_DIR / f"{tag}.zip"
    shutil.copy2(zp, dest)
    os.sync()
    size_mb = Path(zp).stat().st_size / 1e6
    print(f"  >> Checkpoint saved: {dest.name} ({size_mb:.1f} MB, {len(files)} images)")

total = len(SEEDS) * len(selected_prompts)
done = 0

for seed in SEEDS:
    seed_dir = OUTPUT_DIR / f"seed{seed}"
    seed_dir.mkdir(parents=True, exist_ok=True)
    manifest = seed_dir / "manifest.jsonl"
    pending_files = []
    ckpt_count = 6

    for prompt_idx, prompt in selected_prompts:
        t0 = time.time()
        generator = torch.Generator(device=DEVICE).manual_seed(seed)
        result = pipe(
            prompt,
            num_inference_steps=NUM_INFERENCE_STEPS,
            guidance_scale=GUIDANCE_SCALE,
            height=HEIGHT,
            width=WIDTH,
            generator=generator,
        )
        img = result.images[0]
        fname = f"sdxl_prompt{prompt_idx:03d}_seed{seed}.png"
        path = seed_dir / fname
        img.save(path)
        pending_files.append(path)
        del result, img
        torch.cuda.empty_cache()

        with open(manifest, "a") as f:
            f.write(json.dumps({"prompt_id": prompt_idx, "prompt": prompt, "seed": seed, "file": fname}) + "\n")

        done += 1
        elapsed = time.time() - t0
        print(f"[{done}/{total}] seed={seed} prompt={prompt_idx} ({elapsed:.1f}s)")

        if len(pending_files) == CHECKPOINT_EVERY:
            ckpt_count += 1
            checkpoint_to_drive(pending_files, f"sdxl_drawbench_ckpt{ckpt_count}_seed{seed}")
            pending_files = []

    if pending_files:
        ckpt_count += 1
        checkpoint_to_drive(pending_files, f"sdxl_drawbench_ckpt{ckpt_count}_seed{seed}")
        pending_files = []

print(f"\nDone. {done} images saved to {OUTPUT_DIR}")

  0%|          | 0/50 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/diffusers/pipelines/stable_diffusion_xl/pipeline_stable_diffusion_xl.py:748: FutureWarning: `upcast_vae` is deprecated and will be removed in version 1.0.0. `upcast_vae` is deprecated. Please use `pipe.vae.to(torch.float32)`. For more details, please refer to: https://github.com/huggingface/diffusers/pull/12619#issue-3606633695.
  deprecate(


[1/50] seed=2026 prompt=150 (11.6s)


  0%|          | 0/50 [00:00<?, ?it/s]

[2/50] seed=2026 prompt=151 (11.7s)


...

[48/50] seed=2026 prompt=197 (11.5s)


  0%|          | 0/50 [00:00<?, ?it/s]

[49/50] seed=2026 prompt=198 (11.5s)


  0%|          | 0/50 [00:00<?, ?it/s]

[50/50] seed=2026 prompt=199 (11.6s)
  >> Checkpoint saved: sdxl_drawbench_ckpt8_seed2026.zip (37.6 MB, 25 images)

Done. 50 images saved to /content/outputs/daniel_sdxl_drawbench


## Preview

In [ ]:
sample_paths = sorted(OUTPUT_DIR.rglob("*.png"))[-4:]
for p in sample_paths:
    print(p.name)
    display(Image.open(p))

## Copy to Google Drive

In [ ]:
import os
for seed in SEEDS:
    seed_dir = OUTPUT_DIR / f"seed{seed}"
    if not seed_dir.exists():
        print(f"Skipping seed {seed} — no output dir")
        continue
    n = len(list(seed_dir.glob("*.png")))
    tag = f"sdxl_drawbench_{n}imgs_seed{seed}_final"
    zp = shutil.make_archive(f"/content/{tag}", "zip", seed_dir)
    dest = DRIVE_DIR / f"{tag}.zip"
    shutil.copy2(zp, dest)
    print(f"{dest.name}: {Path(zp).stat().st_size / 1e6:.1f} MB")
os.sync()
print("All final zips flushed to Drive.")

In [ ]:
import os
print(os.path.exists("/content/drive/MyDrive"))
print(list(Path("/content/drive/MyDrive/generative-modelling/sdxl_drawbench").iterdir()))

In [ ]:
import shutil
from pathlib import Path

DRIVE_DIR = Path("/content/drive/MyDrive/generative-modelling/sdxl_drawbench")
for z in Path("/content").glob("sdxl_drawbench_batch*.zip"):
    dest = DRIVE_DIR / z.name
    if not dest.exists():
        shutil.copy2(z, dest)
        print(f"Copied {z.name}")

import os
os.sync()
print("Done + flushed")